# 🎙️ CosyVoice Audiobook — GPU-quota-friendly

Render a book in your **cloned voice** with **Fun-CosyVoice 3.0**, structured to **minimize free-GPU usage**.

## Why two parts?
Colab's free GPU is metered by **time connected to a GPU runtime**. So we do all the heavy, GPU-free work (model download, uploads) on a **CPU runtime**, save it to **Google Drive**, and only switch to GPU for the actual voice generation.

| Part | Runtime | Cells | Uses GPU quota? |
|------|---------|-------|:---:|
| **1** | **CPU (None)** | mount Drive · download model → Drive · upload ref+book → Drive | ❌ no |
| **2** | **GPU (T4)** | install · generate (`narrate`) · download | ✅ only here |

Everything persists on Drive, so the model downloads **once ever**, and you can resume the book across many short GPU sessions.

---
### ▶️ START: set **Runtime → Change runtime type → CPU** (None), then run Part 1 below.


In [ ]:
# [ANY runtime] Mount Drive + create folders. Re-run this after switching runtime type.
from google.colab import drive
drive.mount('/content/drive')
import os
BASE   = '/content/drive/MyDrive/cosyvoice_audiobook'
MODELS = os.path.join(BASE, 'models')
OUTPUT = os.path.join(BASE, 'output')
for d in (BASE, MODELS, OUTPUT):
    os.makedirs(d, exist_ok=True)
print("Drive ready:", BASE)


## Part 1 — run on a **CPU runtime** (no GPU used) 👇
Download the model to Drive and upload your files. Do this once; it persists forever on Drive.


In [ ]:
# [CPU ok] Download Fun-CosyVoice 3.0 (~1-2 GB) to DRIVE -- skips if already there.
!pip install -q huggingface_hub
import os
MODEL_DIR = os.path.join(MODELS, 'Fun-CosyVoice3-0.5B')
if os.path.isdir(MODEL_DIR) and any(f.endswith('.safetensors') for f in os.listdir(MODEL_DIR) if os.path.isfile(os.path.join(MODEL_DIR, f))):
    print("model already on Drive -> skipping download:", MODEL_DIR)
else:
    from huggingface_hub import snapshot_download
    snapshot_download('FunAudioLLM/Fun-CosyVoice3-0.5B-2512', local_dir=MODEL_DIR)
    print("downloaded to Drive:", MODEL_DIR)


In [ ]:
# [CPU ok] Upload ref_enhanced.wav + book.txt -> save to DRIVE, then build the 16 kHz prompt.
# Re-run this cell anytime to UPDATE book.txt (re-upload it). ref is kept once uploaded.
import os, shutil, subprocess
REF_DRIVE    = os.path.join(BASE, 'ref_enhanced.wav')
BOOK_DRIVE   = os.path.join(BASE, 'book.txt')
PROMPT_DRIVE = os.path.join(BASE, 'prompt_16k.wav')

need_ref  = not os.path.exists(REF_DRIVE)
from google.colab import files
print("Select files to upload (ref_enhanced.wav if first time, and/or book.txt) ...")
up = files.upload()   # -> /content
for name in up:
    if name.endswith('.wav') and (need_ref or 'ref' in name.lower()):
        shutil.move(name, REF_DRIVE); print("saved ref ->", REF_DRIVE)
    elif name.endswith('.txt'):
        shutil.move(name, BOOK_DRIVE); print("saved book ->", BOOK_DRIVE)

# (re)build the <=15s, 16 kHz prompt on Drive from the reference
if os.path.exists(REF_DRIVE):
    subprocess.run(['ffmpeg','-y','-loglevel','error','-i',REF_DRIVE,'-ar','16000','-ac','1','-t','15',PROMPT_DRIVE], check=True)
    print("prompt_16k.wav ready on Drive")
print("Drive now has:", [f for f in os.listdir(BASE) if os.path.isfile(os.path.join(BASE,f))])


---
## Part 2 — NOW switch to **GPU** 🚀
1. **Runtime → Change runtime type → T4 GPU → Save** (this restarts the VM).
2. **Re-run the Mount-Drive cell above** (a fresh runtime needs Drive re-mounted).
3. Run the cells below (install → engine → narrate).

*Install re-runs each GPU session (packages don't persist a runtime switch) — but the model/files load straight from Drive, so no re-download or re-upload.*


In [ ]:
# [GPU] Confirm a GPU is attached
!nvidia-smi -L || echo "NO GPU -> Runtime > Change runtime type > T4 GPU, then re-run the Mount-Drive cell"


In [ ]:
# [GPU] Clone CosyVoice + install (~5-10 min). Slimmed requirements to avoid Colab dependency conflicts.
!git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git /content/CosyVoice
!cd /content/CosyVoice && git submodule update --init --recursive
!apt-get -qq install -y sox libsox-dev >/dev/null
!grep -vE 'deepspeed|tensorrt|openai-whisper|grpcio|^numpy|^torch|^librosa|^matplotlib|^protobuf|^pyarrow|^tensorboard|^networkx|^onnxruntime' /content/CosyVoice/requirements.txt > /content/reqs_slim.txt && pip install -r /content/reqs_slim.txt && pip uninstall -y onnxruntime-gpu onnxruntime >/dev/null 2>&1; pip install -q openai-whisper 'onnxruntime>=1.19,<2' && pip install -q -U numba llvmlite
!pip install -q modelscope huggingface_hub soundfile
print("install done -- ignore pip 'dependency resolver' WARNINGS about tensorflow/cudf/lightning etc")


### ⏱️ Optional: reduce idle disconnects during the long run
Colab drops a session after **~90 min idle**. A running `narrate(0)` keeps the runtime busy, but if you step away the tab can still disconnect. To keep it alive, open your browser's **DevTools → Console** (press F12) and paste this, then press Enter:

```javascript
setInterval(() => {
  const b = document.querySelector('colab-connect-button');
  (b?.shadowRoot?.querySelector('#connect') || b)?.click();
  console.log('keep-alive', new Date().toLocaleTimeString());
}, 60000);
```

**Notes**
- Clicks the connect button every 60s. Colab's UI changes occasionally; if it logs an error it's harmless.
- It does **not** bypass the ~12 h max-session or GPU-quota limits — nothing can.
- **Your real safety net is resume:** if it disconnects, switch back to GPU, re-run the **Mount-Drive** + **Engine** cells, then `narrate(0)` — it continues from the last finished chunk.


In [ ]:
# [GPU] Narration engine. Paths point at DRIVE so model/book/prompt persist across sessions.
import os, re, glob, time, subprocess, sys, torch
sys.path.append('/content/CosyVoice/third_party/Matcha-TTS')
sys.path.append('/content/CosyVoice')
from cosyvoice.cli.cosyvoice import AutoModel

# ---------------- CONFIG (all on Drive) ----------------
BASE       = '/content/drive/MyDrive/cosyvoice_audiobook'
BOOK_FILE  = os.path.join(BASE, "book.txt")
PROMPT_WAV = os.path.join(BASE, "prompt_16k.wav")            # <=15s, 16 kHz reference
REF_TEXT   = "孩子的成长之路是风雨交加的事实上青春期出现极端情绪是正常的。"   # transcript of the reference clip
MODEL_DIR  = os.path.join(BASE, "models/Fun-CosyVoice3-0.5B")
OUTPUT_DIR = os.path.join(BASE, "output")
MAX_CHARS  = 200
MIN_CHARS  = 40                                             # merge shorter pieces (avoids "text too short" warnings)
SENTENCE_PAUSE  = 0.30
PARAGRAPH_PAUSE = 0.65
MAKE_MP3   = True
# -------------------------------------------------------
CHUNK_DIR = os.path.join(OUTPUT_DIR, "chunks")
PROMPT_TEXT = REF_TEXT if "<|endofprompt|>" in REF_TEXT else REF_TEXT + "<|endofprompt|>"  # CosyVoice3 needs this token

def split_sentences(p):
    p = re.sub(r"\s+", " ", p).strip()
    return [s.strip() for s in re.split(r"(?<=[.!?\u2026\u3002\uff01\uff1f])\s*", p) if s.strip()] if p else []

def split_long(s, m):
    if len(s) <= m: return [s]
    out, buf = [], ""
    for part in re.split(r"(?<=[,;:\uff0c\uff1b\uff1a\u3001])\s*", s):
        if len(part) > m:
            if " " in part:
                for w in part.split(" "):
                    if len(buf)+len(w)+1 > m and buf: out.append(buf.strip()); buf=""
                    buf += w+" "
            else:
                if buf.strip(): out.append(buf.strip()); buf=""
                for i in range(0,len(part),m): out.append(part[i:i+m])
            continue
        if len(buf)+len(part)+1 > m and buf: out.append(buf.strip()); buf=""
        buf += part+" "
    if buf.strip(): out.append(buf.strip())
    return out

def build_chunks(text):
    chunks = []
    for para in [x for x in re.split(r"\n\s*\n", text) if x.strip()]:
        buf, pc = "", []
        for sent in split_sentences(para):
            for piece in split_long(sent, MAX_CHARS):
                if len(buf)+len(piece)+1 > MAX_CHARS and buf: pc.append(buf.strip()); buf=""
                buf += piece+" "
        if buf.strip(): pc.append(buf.strip())
        for i,c in enumerate(pc): chunks.append([c, PARAGRAPH_PAUSE if i==len(pc)-1 else SENTENCE_PAUSE])
    merged = []
    for c, pause in chunks:
        if merged and len(merged[-1][0]) < MIN_CHARS and len(merged[-1][0])+1+len(c) <= MAX_CHARS:
            merged[-1][0] += " " + c; merged[-1][1] = pause
        else:
            merged.append([c, pause])
    return [(c, p) for c, p in merged]

_M = {}
def narrate(test_limit=0):
    import soundfile as sf, numpy as np
    assert os.path.isfile(BOOK_FILE), BOOK_FILE + " not found (do Part 1)"
    assert os.path.isfile(PROMPT_WAV), PROMPT_WAV + " not found (do Part 1)"
    os.makedirs(CHUNK_DIR, exist_ok=True)
    chunks = build_chunks(open(BOOK_FILE, encoding="utf-8").read())
    todo = chunks if test_limit<=0 else chunks[:test_limit]
    print(str(len(chunks)) + " chunks total; rendering " + str(len(todo)))
    if "cv" not in _M:
        _M["cv"] = AutoModel(model_dir=MODEL_DIR)
    cv = _M["cv"]; sr = cv.sample_rate
    t0, done, regen = time.time(), 0, 0
    for i,(ct,pause) in enumerate(todo,1):
        outp = os.path.join(CHUNK_DIR, "chunk_%05d.wav" % i)
        side = os.path.join(CHUNK_DIR, "chunk_%05d.txt" % i)
        if os.path.isfile(outp) and os.path.getsize(outp)>0 and os.path.isfile(side) and open(side, encoding="utf-8").read()==ct:
            continue
        segs = [x["tts_speech"] for x in cv.inference_zero_shot(ct, PROMPT_TEXT, PROMPT_WAV, stream=False)]
        wav = torch.cat(segs, dim=1).squeeze(0).cpu().numpy().astype("float32")
        if pause>0: wav = np.concatenate([wav, np.zeros(int(sr*pause), dtype=np.float32)])
        sf.write(outp, wav, sr, subtype="PCM_16")
        open(side, "w", encoding="utf-8").write(ct)
        done += 1; regen += 1; avg=(time.time()-t0)/done
        print("[%d/%d] %dc  %.1fs/chunk  ETA %.1f min" % (i, len(todo), len(ct), avg, avg*(len(todo)-i)/60))
    for p in glob.glob(os.path.join(CHUNK_DIR, "chunk_*.wav")):
        if int(os.path.basename(p)[6:11]) > len(todo):
            os.remove(p)
            if os.path.exists(p[:-4]+".txt"): os.remove(p[:-4]+".txt")
    paths = [os.path.join(CHUNK_DIR, "chunk_%05d.wav" % i) for i in range(1, len(todo)+1)]
    lst = os.path.join(OUTPUT_DIR, "_concat.txt")
    open(lst, "w").write("".join("file '" + os.path.abspath(p) + "'\n" for p in paths))
    subprocess.run(["ffmpeg","-y","-f","concat","-safe","0","-i",lst,"-c","copy",os.path.join(OUTPUT_DIR,"audiobook.wav")], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if MAKE_MP3:
        subprocess.run(["ffmpeg","-y","-f","concat","-safe","0","-i",lst,"-c:a","libmp3lame","-q:a","4",os.path.join(OUTPUT_DIR,"audiobook.mp3")], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print("DONE (%d newly generated) -> %s" % (regen, os.path.join(OUTPUT_DIR, "audiobook.mp3")))

print("ready. Run narrate(test_limit=5) for a smoke test, then narrate(test_limit=0) for the whole book.")


In [ ]:
# [GPU] Smoke test -- first 5 chunks
narrate(test_limit=5)


In [ ]:
# [GPU] Full book (auto-resumes if it disconnects -- just re-run this cell)
narrate(test_limit=0)


In [ ]:
# Download the finished audiobook (it also lives permanently in your Drive output/ folder)
from google.colab import files
files.download('/content/drive/MyDrive/cosyvoice_audiobook/output/audiobook.mp3')
